# 04 Discrete Choice Model Versus Prediction Benchmark

## Program Overview

This program provides a compact teaching version of the MOOC DCM-versus-ML idea. It compares an interpretable Logit model with a feature-rich benchmark and explains how to read validation results.

## Overall Task AI Prompt (English)

Use this notebook as a programming lesson for discrete choice modeling. Before
running each code cell, read the prompt and explain what theoretical concept the
code implements. After running the cell, use the interpretation cell to connect
the numerical result back to utility, probability, likelihood, estimation, or
model validation.

## Learning Objectives

- Define the decision maker, alternatives, choice variable, and explanatory attributes.
- Translate random utility theory into executable Python code.
- Interpret model probabilities, coefficients, likelihood values, and validation results.
- Distinguish behavioral explanation from prediction accuracy.

## AI Prompt for This Code Cell

Create a mode-choice data set and split it into training and test sets. The split mirrors the validation logic in the MOOC DCM-ML notebooks.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 180
time_diff = rng.normal(0, 10, n)
cost_diff = rng.normal(1.5, 2.5, n)
transfer_penalty = rng.integers(0, 3, n)

utility = 0.20 - 0.09 * time_diff - 0.50 * cost_diff - 0.35 * transfer_penalty
prob_car = 1.0 / (1.0 + np.exp(-utility))
choice_car = rng.binomial(1, prob_car)

data = pd.DataFrame({
    "time_diff": time_diff,
    "cost_diff": cost_diff,
    "transfer_penalty": transfer_penalty,
    "choice_car": choice_car,
})

train = data.sample(frac=0.7, random_state=1)
test = data.drop(train.index)

print(f"Training observations: {len(train)}")
print(f"Test observations: {len(test)}")
data.head()

Training observations: 126
Test observations: 54


,time_diff,cost_diff,transfer_penalty,choice_car
0,0.012302,0.397137,2,0
1,2.987455,0.230098,2,1
2,-2.741379,3.075206,0,0
3,-8.905918,0.745331,2,0
4,-4.546708,1.121391,1,1


## Output Interpretation

The output reports 126 training observations and 54 test observations. This split is important because the model coefficients are learned from the training data, while prediction quality is evaluated on unseen test data. The data include `time_diff`, `cost_diff`, and `transfer_penalty`. The first two variables are standard transport-choice attributes, while the transfer feature is included to mimic the richer predictor sets often used in machine learning comparisons. Students should see this as a validation setup rather than a final empirical study.


## AI Prompt for This Code Cell

Estimate an interpretable binary Logit model using only time and cost. This represents the discrete choice modeling baseline.

In [2]:
def fit_logit_gradient(df, columns, steps=6000, learning_rate=0.0007):
    X = np.column_stack([np.ones(len(df))] + [df[col].to_numpy() for col in columns])
    y = df["choice_car"].to_numpy()
    beta = np.zeros(X.shape[1])
    for _ in range(steps):
        p = 1.0 / (1.0 + np.exp(-(X @ beta)))
        beta -= learning_rate * (X.T @ (p - y))
    return beta

def predict_logit(df, columns, beta):
    X = np.column_stack([np.ones(len(df))] + [df[col].to_numpy() for col in columns])
    return 1.0 / (1.0 + np.exp(-(X @ beta)))

dcm_columns = ["time_diff", "cost_diff"]
dcm_beta = fit_logit_gradient(train, dcm_columns)
dcm_summary = pd.Series(dcm_beta, index=["intercept"] + dcm_columns)
dcm_summary

intercept   -0.504157
time_diff   -0.099289
cost_diff   -0.484404
dtype: float64

## Output Interpretation

The interpretable DCM estimates are approximately `intercept = -0.5042`, `time_diff = -0.0993`, and `cost_diff = -0.4844`. Both slope coefficients have the expected negative signs: when car becomes slower or more expensive relative to the competing mode, the probability of choosing car decreases. The negative intercept indicates that, when time and cost differences are zero, the fitted baseline utility for car is lower than the competing alternative in this training sample. This is the behavioral advantage of the DCM: each coefficient can be discussed as a sensitivity to a transport attribute.


## AI Prompt for This Code Cell

Fit a more flexible prediction model by adding transfer penalty. This is a simple ML-style benchmark because it prioritizes predictive features rather than a minimal behavioral specification.

In [3]:
ml_columns = ["time_diff", "cost_diff", "transfer_penalty"]
ml_beta = fit_logit_gradient(train, ml_columns)
ml_summary = pd.Series(ml_beta, index=["intercept"] + ml_columns)
ml_summary

intercept          -0.088930
time_diff          -0.106439
cost_diff          -0.485345
transfer_penalty   -0.409889
dtype: float64

## Output Interpretation

The feature-rich benchmark estimates are approximately `intercept = -0.0889`, `time_diff = -0.1064`, `cost_diff = -0.4853`, and `transfer_penalty = -0.4099`. The time and cost effects remain close to the DCM estimates, which suggests that adding the transfer variable does not overturn the main behavioral pattern. The negative transfer coefficient means additional transfer burden lowers the probability of choosing car in this synthetic setup. Compared with the DCM, the intercept becomes less negative because part of the baseline difference is now explained by the transfer variable. This is a useful example of how adding predictors can change coefficient interpretation.


## AI Prompt for This Code Cell

Compare the two models on the test set using accuracy and average log-likelihood.

In [4]:
def evaluate(probability, observed):
    probability = np.clip(probability, 1e-12, 1 - 1e-12)
    prediction = (probability >= 0.5).astype(int)
    accuracy = (prediction == observed).mean()
    avg_log_likelihood = np.mean(
        observed * np.log(probability) + (1 - observed) * np.log(1 - probability)
    )
    return accuracy, avg_log_likelihood

y_test = test["choice_car"].to_numpy()
dcm_prob = predict_logit(test, dcm_columns, dcm_beta)
ml_prob = predict_logit(test, ml_columns, ml_beta)

results = pd.DataFrame(
    {
        "model": ["Interpretable DCM", "Feature-rich benchmark"],
        "accuracy": [evaluate(dcm_prob, y_test)[0], evaluate(ml_prob, y_test)[0]],
        "average_log_likelihood": [
            evaluate(dcm_prob, y_test)[1],
            evaluate(ml_prob, y_test)[1],
        ],
    }
)

results

,model,accuracy,average_log_likelihood
0,Interpretable DCM,0.759259,-0.496315
1,Feature-rich benchmark,0.814815,-0.462074


## Output Interpretation

On the test set, the interpretable DCM has accuracy 75.93% and average log-likelihood -0.4963. The feature-rich benchmark improves to accuracy 81.48% and average log-likelihood -0.4621. Because higher accuracy is better and log-likelihood closer to zero is better, the benchmark predicts better on both metrics. The teaching point is not that the benchmark is automatically superior; it is that predictive performance and behavioral interpretation must be balanced. The DCM is easier to explain in terms of utility theory, while the feature-rich model uses additional information to improve out-of-sample prediction.
